# HRN Plugin Analysis

This notebook compares HRN baselines against HRN runs wrapped with the shared orthonormal taxonomy-frame plugin.

Expected output directories:

- baselines: `hrn_<dataset>`
- plugin runs: `hrn_<dataset>_orthonormal_plugin_<loss>_baseline_<weight>`


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / 'configs').exists() else cwd.parent
notebook_utils_dir = repo_root / 'notebooks'
if str(notebook_utils_dir) not in sys.path:
    sys.path.insert(0, str(notebook_utils_dir))

from hcast_analysis_utils import (
    HCastAnalysis as HRNAnalysis,
    HCastAnalysisConfig as HRNAnalysisConfig,
    resolve_project_root,
)

PROJECT_ROOT = resolve_project_root()
OUTPUTS_ROOT = Path('/scratch/g.saggini1/outputs')
print(f'Using project root: {PROJECT_ROOT}')


In [ ]:
BASELINE_COLOR = '#243ab4'

DATASET_SLUGS = {
    'cifar-100': 'cifar100',
    'cub-200-2011': 'cub200',
    'fgvc-aircraft': 'aircraft',
}
HRN_DATASETS = tuple(DATASET_SLUGS.values())

PLUGIN_LOSS_MODE = 'global_softmax_ce_reg'
PLUGIN_LOSS_LABEL = 'global softmax CE+reg'
PLUGIN_WEIGHT_MODE = 'equal'

BASELINE_RUNS = [
    {
        'run_dir': f'hrn_{dataset}',
        'label': 'HRN baseline',
        'is_baseline': True,
        'color': BASELINE_COLOR,
    }
    for dataset in HRN_DATASETS
]

PLUGIN_RUNS = [
    {
        'run_dir': f'hrn_{dataset}_orthonormal_plugin_{PLUGIN_LOSS_MODE}_baseline_{PLUGIN_WEIGHT_MODE}',
        'label': f'HRN plugin {PLUGIN_LOSS_LABEL} {PLUGIN_WEIGHT_MODE}',
    }
    for dataset in HRN_DATASETS
]

MANUAL_RUNS = BASELINE_RUNS + PLUGIN_RUNS

TEMPERATURE_COLOR_PALETTE = [
    '#d62728',
    '#2ca02c',
    '#ff7f0e',
    '#9467bd',
    '#8c564b',
    '#17becf',
    '#e377c2',
    '#bcbd22',
]


In [ ]:
config = HRNAnalysisConfig(
    outputs_root=OUTPUTS_ROOT,
    include_baselines=False,
    baseline_color=BASELINE_COLOR,
    manual_runs=MANUAL_RUNS,
    temperature_color_palette=TEMPERATURE_COLOR_PALETTE,
)

analysis = HRNAnalysis.from_config(config)
analysis.print_run_summary()


## Validation Curves

- color = run
- line style = mode (`topdown` solid, `independent` dashed)

Metrics shown: `FPA`, `wAP`, `TICE`, and `AHD`.


In [ ]:
analysis.plot_validation_curves()


## HRN Consistency Diagnostics

These panels use HRN's logged level-2 projection/parent-consistency diagnostics when present.


In [ ]:
analysis.plot_projection_diagnostics(
    base_diag_specs=[
        ('proj_flip_rate_level_2', 'Level-2 projection flip rate', True, 'val'),
        ('proj_delta_l1_level_2', 'Level-2 projection L1 delta', False, 'val'),
        ('proj_gt_prob_delta_level_2', 'Level-2 GT probability delta', False, 'val'),
        ('gt_parent_mass_pre_l2', 'GT parent mass before level-2 projection', False, 'val'),
        ('gt_parent_mass_post_l2', 'GT parent mass after level-2 projection', False, 'val'),
    ]
)


## Training Loss Comparison

The baseline HRN losses are `tree_loss`, `hier_loss`, `ce_loss_leaf`, and `fine_ce`; plugin runs can additionally expose `ce`, `reg`, `kl`, and `loss_level_*` terms.


In [ ]:
analysis.plot_training_losses(
    aggregate_loss_keys=['total', 'tree_loss', 'hier_loss', 'ce_loss_leaf', 'fine_ce', 'ce', 'reg', 'kl']
)


## Per-Run Per-Level Training Losses

This is mainly useful for plugin runs because the orthonormal plugin exposes per-level differentiable loss terms.


In [ ]:
analysis.plot_per_run_per_level_training_losses()


## Per-Level Validation Accuracy

Top-down and independent decoding are shown separately for each hierarchy level.


In [ ]:
analysis.plot_per_level_validation_accuracy()


## Final Test Comparison Table

For each dataset, metrics are rows and runs are columns. Non-baseline cells show deltas versus HRN baseline.


In [ ]:
analysis.show_final_test_tables()
